# Imports

In [39]:
import os
import time
import gc
import sys
import numpy as np
import pandas as pd
import psutil
import requests
from PIL import Image
from tqdm.notebook import tqdm

In [42]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from transformers import (
    ViTModel, ViTImageProcessor,
    AutoImageProcessor, AutoModel,
    DeiTModel, DeiTImageProcessor,
    CLIPProcessor, CLIPVisionModel, CLIPTextModel,
    BertTokenizer, BertModel, 
    RobertaTokenizer, RobertaModel,
    GPT2Tokenizer, GPT2Model
)

# Configuration

In [65]:
#("Flickr8k", "Flickr30k", "ConceptualCaptions")
CURRENT_DATASET = "Flickr30k" 

# --- DIRECTORY ARCHITECTURE ---
DATA_DIR = os.path.join(os.getcwd(), 'TFE_Data')
DATASETS_DIR = os.path.join(DATA_DIR, 'Datasets')
RAW_DIR = os.path.join(DATA_DIR, 'Features_RAW')
RESULTS_DIR = os.path.join(DATA_DIR, 'Results')

for d in [DATASETS_DIR, RAW_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

# Device Configuration
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Environment initialized. Using device: {device}")

Environment initialized. Using device: cuda


# Data Loading

In [29]:
print(f"Loading dataset: {CURRENT_DATASET}...")

df_path = os.path.join(DATASETS_DIR, f"df_{CURRENT_DATASET}.pkl")
if not os.path.exists(df_path):
    raise FileNotFoundError(f"Dataset file not found at {df_path}. Please run data_builder.ipynb first.")

df = pd.read_pickle(df_path)

# Extract inputs for models
IMAGE_PATHS = df['image_path'].tolist()
ALIGNED_CAPTIONS = [caps[0] for caps in df['captions']]

print(f"Ready. Loaded {len(IMAGE_PATHS)} images and {len(ALIGNED_CAPTIONS)} captions into memory.")

Loading dataset: Flickr8k...
Ready. Loaded 8091 images and 8091 captions into memory.


# Utility Functions
## GreenAI Metrics

In [61]:
def send_ntfy_notification(message, title="TFE Pipeline Update"):
    """Sends a push notification via ntfy.sh."""
    try:
        requests.post(
            f"https://ntfy.sh/{NTFY_TOPIC}",
            data=message.encode(encoding='utf-8'),
            headers={"Title": title}
        )
    except Exception as e:
        print(f"Notification failed: {e}")

def measure_memory():
    """Returns the current memory usage of the process in MB."""
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / (1024 * 1024)

def get_size_in_mb(obj):
    """Returns the size of an object in MB."""
    return sys.getsizeof(obj) / (1024 * 1024)



In [62]:
def execute_and_save(modality, model_name, extract_func, data, device):
    """Measures execution metrics, extracts features, and saves them to disk."""
    print(f"\nProcessing {modality.upper()} with model: {model_name}")
    
    start_time = time.time()
    mem_before = measure_memory()
    
    features = extract_func(data, device)
    
    end_time = time.time()
    mem_after = measure_memory()
    
    exec_time = end_time - start_time
    mem_used = max(0, mem_after - mem_before)
    original_size_mb = get_size_in_mb(features)
    
    # Extract dimension BEFORE deleting the features array
    original_dim = features.shape[1] if hasattr(features, 'shape') else 0
    
    save_path = os.path.join(RAW_DIR, f"X_{modality}_{model_name}_raw_{CURRENT_DATASET}.npy")
    np.save(save_path, features)
    
    print(f"Extraction complete. Shape: {features.shape}. Time: {exec_time:.2f}s, Memory Delta: {mem_used:.2f} MB")
    print(f"Saved RAW features to: {save_path}")
    
    # Force garbage collection to free up memory before the next model loads
    del features
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        
    return {
        "Modality": modality,
        "Model": model_name,
        "Original_Dim": original_dim,
        "Exec_Time_s": exec_time,
        "Memory_Used_MB": mem_used,
        "Original_Size_MB": original_size_mb
    }

# Unimodal Models

## CBIR Vision

### Indexing: Embedding Models

In [ ]:
def get_resnet50_embeddings(paths, device):
    """Extracts RAW visual features using heavy CNN (ResNet50)."""
    weights = models.ResNet50_Weights.DEFAULT
    model = models.resnet50(weights=weights).to(device).eval()
    model.fc = nn.Identity() # Strip classification head
    transform = weights.transforms()
    
    embeddings = []
    with torch.no_grad():
        for path in tqdm(paths, desc="Extracting ResNet50"):
            img = Image.open(path).convert('RGB')
            tensor = transform(img).unsqueeze(0).to(device)
            embeddings.append(model(tensor).squeeze().cpu().numpy())
    return model, np.array(embeddings)

In [11]:
def get_mobilenet_v3_embeddings(paths, device):
    """Extracts RAW visual features using lightweight CNN (MobileNetV3)."""
    weights = models.MobileNet_V3_Small_Weights.DEFAULT
    model = models.mobilenet_v3_small(weights=weights).to(device).eval()
    model.classifier[3] = nn.Identity() # Strip classification head
    transform = weights.transforms()
    
    embeddings = []
    with torch.no_grad():
        for path in tqdm(paths, desc="Extracting MobileNetV3"):
            img = Image.open(path).convert('RGB')
            tensor = transform(img).unsqueeze(0).to(device)
            embeddings.append(model(tensor).squeeze().cpu().numpy())
    return model, np.array(embeddings)

In [12]:
def get_vit_embeddings(paths, device):
    """Extracts RAW visual features using standard Transformer (ViT)."""
    processor = ViTImageProcessor.from_pretrained('google/vit-base-patch16-224-in21k')
    model = ViTModel.from_pretrained('google/vit-base-patch16-224-in21k').to(device).eval()
    
    embeddings = []
    with torch.no_grad():
        for path in tqdm(paths, desc="Extracting ViT"):
            img = Image.open(path).convert('RGB')
            inputs = processor(images=img, return_tensors="pt").to(device)
            # Pooler output represents the [CLS] token global representation
            embeddings.append(model(**inputs).pooler_output.squeeze().cpu().numpy())
    return model, np.array(embeddings)

In [13]:
def get_deit_embeddings(paths, device):
    """Extracts RAW visual features using data-efficient Transformer (DeiT)."""
    processor = DeiTImageProcessor.from_pretrained('facebook/deit-base-distilled-patch16-224')
    model = DeiTModel.from_pretrained('facebook/deit-base-distilled-patch16-224').to(device).eval()
    
    embeddings = []
    with torch.no_grad():
        for path in tqdm(paths, desc="Extracting DeiT"):
            img = Image.open(path).convert('RGB')
            inputs = processor(images=img, return_tensors="pt").to(device)
            # Extract the first token representation (similar to [CLS])
            embeddings.append(model(**inputs).last_hidden_state[:, 0, :].squeeze().cpu().numpy())
    return model, np.array(embeddings)

In [14]:
def get_clip_vision_embeddings(paths, device):
    """Extracts RAW visual features using natively multimodal encoder (CLIP Vision)."""
    processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
    model = CLIPVisionModel.from_pretrained("openai/clip-vit-base-patch32").to(device).eval()
    
    embeddings = []
    with torch.no_grad():
        for path in tqdm(paths, desc="Extracting CLIP Vision"):
            img = Image.open(path).convert('RGB')
            inputs = processor(images=img, return_tensors="pt").to(device)
            embeddings.append(model(**inputs).pooler_output.squeeze().cpu().numpy())
    return model, np.array(embeddings)

## Vision Feature Extractions

In [58]:
class ImageDataset(torch.utils.data.Dataset):
    def __init__(self, image_paths, transform=None):
        self.image_paths = image_paths
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        try:
            image = Image.open(img_path).convert('RGB')
        except Exception as e:
            print(f"Error loading image {img_path}: {e}")
            image = Image.new('RGB', (224, 224), (0, 0, 0)) 
        if self.transform:
            image = self.transform(image)
        return image

def extract_vision_features(model, dataloader, device):
    model.eval()
    features = []
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Extracting Vision"):
            batch = batch.to(device)
            
            if hasattr(model, 'get_image_features'): # CLIP logic
                outputs = model.get_image_features(pixel_values=batch)
            else:
                outputs = model(batch)
            
            # Output handling safeguard
            if hasattr(outputs, 'pooler_output') and outputs.pooler_output is not None:
                batch_features = outputs.pooler_output.cpu().numpy()
            elif hasattr(outputs, 'last_hidden_state'):
                batch_features = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            elif hasattr(outputs, 'image_embeds'):
                batch_features = outputs.image_embeds.cpu().numpy()
            else:
                batch_features = outputs.cpu().numpy()
                
            features.append(batch_features)
    return np.vstack(features)

def get_resnet50_embeddings(image_paths, device):
    weights = models.ResNet50_Weights.DEFAULT
    model = models.resnet50(weights=weights).to(device)
    model.fc = nn.Identity()
    dataset = ImageDataset(image_paths, transform=weights.transforms())
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=32, num_workers=4)
    return extract_vision_features(model, dataloader, device)

def get_mobilenet_v3_embeddings(image_paths, device):
    weights = models.MobileNet_V3_Large_Weights.DEFAULT
    model = models.mobilenet_v3_large(weights=weights).to(device)
    model.classifier = nn.Identity()
    dataset = ImageDataset(image_paths, transform=weights.transforms())
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=32, num_workers=4)
    return extract_vision_features(model, dataloader, device)

def get_vit_embeddings(image_paths, device):
    processor = ViTImageProcessor.from_pretrained('google/vit-base-patch16-224-in21k')
    model = ViTModel.from_pretrained('google/vit-base-patch16-224-in21k').to(device)
    def transform(img): return processor(images=img, return_tensors="pt")['pixel_values'].squeeze(0)
    dataset = ImageDataset(image_paths, transform=transform)
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=32, num_workers=4)
    return extract_vision_features(model, dataloader, device)

def get_deit_embeddings(image_paths, device):
    processor = AutoImageProcessor.from_pretrained('facebook/deit-base-distilled-patch16-224')
    model = AutoModel.from_pretrained('facebook/deit-base-distilled-patch16-224').to(device)
    def transform(img): return processor(images=img, return_tensors="pt")['pixel_values'].squeeze(0)
    dataset = ImageDataset(image_paths, transform=transform)
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=32, num_workers=4)
    return extract_vision_features(model, dataloader, device)

def get_clip_vision_embeddings(image_paths, device):
    from transformers import CLIPModel, CLIPProcessor
    model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
    processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
    def transform(img): return processor(images=img, return_tensors="pt")['pixel_values'].squeeze(0)
    dataset = ImageDataset(image_paths, transform=transform)
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=32, num_workers=4)
    return extract_vision_features(model, dataloader, device)

## T2T Text 

### Indexing: Embedding Models

In [15]:
# Custom BiLSTM Architecture for the RNN Baseline
class BiLSTMEncoder(nn.Module):
    def __init__(self, vocab_size, embed_dim=300, hidden_dim=384): 
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
    def forward(self, x):
        embedded = self.embedding(x)
        _, (hidden, _) = self.lstm(embedded)
        # Concatenate forward and backward final hidden states (384 * 2 = 768 dimensions)
        return torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)

In [16]:
def get_rnn_embeddings(texts, device):
    """Extracts RAW semantic features using sequential modeling (BiLSTM)."""
    tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
    model = BiLSTMEncoder(vocab_size=tokenizer.vocab_size).to(device).eval()
    
    embeddings = []
    with torch.no_grad():
        for text in tqdm(texts, desc="Extracting RNN (BiLSTM)"):
            inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True)['input_ids'].to(device)
            embeddings.append(model(inputs).squeeze().cpu().numpy())
    return model, np.array(embeddings)

In [17]:
def get_bert_embeddings(texts, device):
    """Extracts RAW semantic features using bidirectional Auto-Encoder (BERT)."""
    tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
    model = BertModel.from_pretrained('bert-base-uncased').to(device).eval()
    
    embeddings = []
    with torch.no_grad():
        for text in tqdm(texts, desc="Extracting BERT"):
            inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True).to(device)
            embeddings.append(model(**inputs).pooler_output.squeeze().cpu().numpy())
    return model, np.array(embeddings)

In [18]:
def get_roberta_embeddings(texts, device):
    """Extracts RAW semantic features using optimized Auto-Encoder (RoBERTa)."""
    tokenizer = RobertaTokenizer.from_pretrained('roberta-base')
    model = RobertaModel.from_pretrained('roberta-base').to(device).eval()
    
    embeddings = []
    with torch.no_grad():
        for text in tqdm(texts, desc="Extracting RoBERTa"):
            inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True).to(device)
            embeddings.append(model(**inputs).pooler_output.squeeze().cpu().numpy())
    return model, np.array(embeddings)

In [19]:
def get_gpt2_embeddings(texts, device):
    """Extracts RAW semantic features using auto-regressive Decoder (GPT-2)."""
    tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
    tokenizer.pad_token = tokenizer.eos_token
    model = GPT2Model.from_pretrained('gpt2').to(device).eval()
    
    embeddings = []
    with torch.no_grad():
        for text in tqdm(texts, desc="Extracting GPT-2"):
            inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True).to(device)
            # Use the last hidden state of the last token for causal models
            embeddings.append(model(**inputs).last_hidden_state[:, -1, :].squeeze().cpu().numpy())
    return model, np.array(embeddings)

In [20]:
def get_clip_text_embeddings(texts, device):
    """Extracts RAW semantic features using natively multimodal encoder (CLIP Text)."""
    processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
    model = CLIPTextModel.from_pretrained("openai/clip-vit-base-patch32").to(device).eval()
    
    embeddings = []
    with torch.no_grad():
        for text in tqdm(texts, desc="Extracting CLIP Text"):
            inputs = processor(text=text, return_tensors="pt", padding=True, truncation=True).to(device)
            embeddings.append(model(**inputs).pooler_output.squeeze().cpu().numpy())
    return model, np.array(embeddings)

## Text Feature Extraction

In [59]:
class TextDataset(torch.utils.data.Dataset):
    def __init__(self, texts, tokenizer, max_length=128):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        encoding = self.tokenizer(
            text, truncation=True, padding='max_length',
            max_length=self.max_length, return_tensors='pt'
        )
        return {key: val.squeeze(0) for key, val in encoding.items()}

def extract_text_features(model, dataloader, device, feature_type='last_hidden_state_mean'):
    model.eval()
    features = []
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Extracting Text"):
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            
            if feature_type == 'last_hidden_state_mean':
                attention_mask = batch['attention_mask'].unsqueeze(-1).expand(outputs.last_hidden_state.size()).float()
                sum_embeddings = torch.sum(outputs.last_hidden_state * attention_mask, 1)
                sum_mask = torch.clamp(attention_mask.sum(1), min=1e-9)
                batch_features = (sum_embeddings / sum_mask).cpu().numpy()
            else:
                batch_features = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            features.append(batch_features)
    return np.vstack(features)

class SimpleRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim=256, hidden_dim=512):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.rnn = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
    def forward(self, input_ids, attention_mask=None):
        embedded = self.embedding(input_ids)
        output, (hidden, cell) = self.rnn(embedded)
        return hidden[-1]

def get_rnn_embeddings(texts, device):
    tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
    model = SimpleRNN(tokenizer.vocab_size).to(device)
    dataset = TextDataset(texts, tokenizer)
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=32)
    
    model.eval()
    features = []
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Extracting RNN"):
            output = model(batch['input_ids'].to(device))
            features.append(output.cpu().numpy())
    return np.vstack(features)

def get_bert_embeddings(texts, device):
    tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
    model = BertModel.from_pretrained('bert-base-uncased').to(device)
    dataset = TextDataset(texts, tokenizer)
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=32)
    return extract_text_features(model, dataloader, device)

def get_roberta_embeddings(texts, device):
    tokenizer = RobertaTokenizer.from_pretrained('roberta-base')
    model = RobertaModel.from_pretrained('roberta-base').to(device)
    dataset = TextDataset(texts, tokenizer)
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=32)
    return extract_text_features(model, dataloader, device)

def get_gpt2_embeddings(texts, device):
    tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
    tokenizer.pad_token = tokenizer.eos_token
    model = GPT2Model.from_pretrained('gpt2').to(device)
    dataset = TextDataset(texts, tokenizer)
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=32)
    return extract_text_features(model, dataloader, device)

def get_clip_text_embeddings(texts, device):
    from transformers import CLIPModel, CLIPProcessor
    model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
    processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
    
    features = []
    model.eval()
    with torch.no_grad():
        batch_size = 32
        for i in tqdm(range(0, len(texts), batch_size), desc="Extracting CLIP Text"):
            batch_texts = texts[i:i+batch_size]
            inputs = processor(text=batch_texts, return_tensors="pt", padding=True, truncation=True).to(device)
            outputs = model.get_text_features(**inputs)
            
            if hasattr(outputs, 'text_embeds'):
                batch_features = outputs.text_embeds.cpu().numpy()
            elif hasattr(outputs, 'pooler_output'):
                batch_features = outputs.pooler_output.cpu().numpy()
            else:
                batch_features = outputs.cpu().numpy()
            features.append(batch_features)
    return np.vstack(features)

### Execution

In [66]:
green_metrics = []

vision_pipeline = {
    "resnet50": get_resnet50_embeddings,
    "mobilenet_v3": get_mobilenet_v3_embeddings,
    "vit": get_vit_embeddings,
    "deit": get_deit_embeddings,
    "clip_vision": get_clip_vision_embeddings
}

text_pipeline = {
    "rnn": get_rnn_embeddings,
    "bert": get_bert_embeddings,
    "roberta": get_roberta_embeddings,
    "gpt2": get_gpt2_embeddings,
    "clip_text": get_clip_text_embeddings
}

send_ntfy_notification(f"Pipeline started for dataset: {CURRENT_DATASET}", "TFE Indexation Started")

print("\n" + "="*40 + "\nINITIATING VISION PIPELINE\n" + "="*40)
for name, func in vision_pipeline.items():
    metrics = execute_and_save("vision", name, func, IMAGE_PATHS, device)
    green_metrics.append(metrics)
    
send_ntfy_notification(f"Vision pipeline completed for {CURRENT_DATASET}.", "TFE Progress Update")

print("\n" + "="*40 + "\nINITIATING TEXT PIPELINE\n" + "="*40)
for name, func in text_pipeline.items():
    metrics = execute_and_save("text", name, func, ALIGNED_CAPTIONS, device)
    green_metrics.append(metrics)
    
# Save consolidated metrics
df_green = pd.DataFrame(green_metrics)
green_metrics_path = os.path.join(RESULTS_DIR, f'unimodal_green_metrics_RAW_{CURRENT_DATASET}.pkl')
df_green.to_pickle(green_metrics_path)

send_ntfy_notification(
    f"SUCCESS: Full unimodal indexation finished for {CURRENT_DATASET}. Metrics saved.", 
    "TFE Pipeline Finished"
)

print(f"\nPipeline successfully completed. Green Metrics saved to {green_metrics_path}")
display(df_green)


INITIATING VISION PIPELINE

Processing VISION with model: resnet50


Extracting Vision:   0%|          | 0/253 [00:00<?, ?it/s]

Extraction complete. Shape: (8091, 2048). Time: 7.51s, Memory Delta: 50.57 MB
Saved RAW features to: /home/aysel/tfe/TFE_Data/Features_RAW/X_vision_resnet50_raw_Flickr30k.npy

Processing VISION with model: mobilenet_v3


Extracting Vision:   0%|          | 0/253 [00:00<?, ?it/s]

Extraction complete. Shape: (8091, 960). Time: 7.39s, Memory Delta: 23.63 MB
Saved RAW features to: /home/aysel/tfe/TFE_Data/Features_RAW/X_vision_mobilenet_v3_raw_Flickr30k.npy

Processing VISION with model: vit


...

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Extracting CLIP Text:   0%|          | 0/253 [00:00<?, ?it/s]

Extraction complete. Shape: (8091, 512). Time: 5.06s, Memory Delta: 2.36 MB
Saved RAW features to: /home/aysel/tfe/TFE_Data/Features_RAW/X_text_clip_text_raw_Flickr30k.npy

Pipeline successfully completed. Green Metrics saved to /home/aysel/tfe/TFE_Data/Results/unimodal_green_metrics_RAW_Flickr30k.pkl


,Modality,Model,Original_Dim,Exec_Time_s,Memory_Used_MB,Original_Size_MB
0,vision,resnet50,2048,7.505592,50.570312,63.211060
1,vision,mobilenet_v3,960,7.389122,23.628906,29.630249
2,vision,vit,768,21.207031,0.003906,23.704224
3,vision,deit,768,22.832804,0.000000,23.704224
4,vision,clip_vision,512,12.497366,3.218750,15.802856
5,text,rnn,512,2.593572,0.000000,15.802856
6,text,bert,768,14.224347,0.003906,23.704224
7,text,roberta,768,13.980452,5.285156,23.704224
8,text,gpt2,768,15.909953,5.320312,23.704224
9,text,clip_text,512,5.057157,2.355469,15.802856


In [56]:
# Clean up duplicate runs by keeping only the last successful extraction for each model
df_green = df_green.drop_duplicates(subset=['Modality', 'Model'], keep='last').reset_index(drop=True)

# Re-save the clean table
df_green.to_pickle(green_metrics_path)

print("Cleaned Green Metrics (10 Models):")
display(df_green)

Cleaned Green Metrics (10 Models):


,Modality,Model,Original_Dim,Exec_Time_s,Memory_Used_MB,Original_Size_MB
0,vision,resnet50,2048,7.457824,41.671875,63.211060
1,vision,mobilenet_v3,960,7.068643,29.082031,29.630249
2,vision,vit,768,20.725187,0.003906,23.704224
3,vision,deit,768,21.785885,0.000000,23.704224
4,vision,clip_vision,512,12.452065,9.835938,15.802856
5,text,rnn,512,2.638184,0.000000,15.802856
6,text,bert,768,13.716853,0.000000,23.704224
7,text,roberta,768,13.266207,3.269531,23.704224
8,text,gpt2,768,15.428872,1.367188,23.704224
9,text,clip_text,512,4.925511,5.585938,15.802856
